# Qwen3.5 Baseline

Notebook baseline cho `ViTextVQA` / `ViOCRVQA` với 2 metric `EM` và `F1`.

Notebook này giữ cùng flow đánh giá như `Qwen2VL.ipynb`, nhưng chuyển sang nhóm model `Qwen3.5` có số tham số `<= 8B` trong phạm vi model công khai hiện có (`0.8B`, `2B`, `4B`).

Sampling mặc định bám theo khuyến nghị `non-thinking VL` của Qwen3.5; riêng `presence_penalty` không được truyền vì `transformers.generate()` chưa hỗ trợ tham số đó theo cách tổng quát.

Lưu ý: notebook này cần `transformers>=5.2.0` để có class `Qwen3_5ForConditionalGeneration` hoặc fallback auto-model tương ứng.

In [ ]:
import os
from pathlib import Path

from huggingface_hub import hf_hub_download

REPO_ROOT = Path('/content/CS2230-VQA')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / 'baseline').exists() and (REPO_ROOT.parent / 'baseline').exists():
        REPO_ROOT = REPO_ROOT.parent

repo_id = 'minhquan6203/ViTextVQA'
filenames_to_download = [
    'ViTextVQA_train.json',
    'ViTextVQA_dev.json',
    'ViTextVQA_test_gt.json',
    'ViTextVQA_images.zip',
]
folder = REPO_ROOT / 'data' / 'ViTextVQA'
folder.mkdir(parents=True, exist_ok=True)

for filename in filenames_to_download:
    try:
        downloaded_file_path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            repo_type='dataset',
            local_dir=str(folder),
        )
        print(f'Downloaded {filename} -> {downloaded_file_path}')
    except Exception as exc:
        print(f'Failed to download {filename}: {exc}')


In [ ]:
from zipfile import ZipFile

zip_path = REPO_ROOT / 'data' / 'ViTextVQA' / 'ViTextVQA_images.zip'
extract_dir = REPO_ROOT / 'data' / 'ViTextVQA'

if zip_path.exists():
    with ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f'Extracted -> {extract_dir}')
else:
    print(f'Zip file not found: {zip_path}')


In [ ]:
from pathlib import Path

# =========================
# Central Config
# =========================
REPO_ROOT = Path('/content/CS2230-VQA')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / 'baseline').exists() and (REPO_ROOT.parent / 'baseline').exists():
        REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / 'data'
DATASET_NAME = 'ViTextVQA'  # 'ViTextVQA' | 'ViOCRVQA'
DATASET_DIR = DATA_ROOT / DATASET_NAME
DEFAULT_DATASET_LAYOUT = {
    'ViTextVQA': {
        'image_dir': DATASET_DIR / 'st_images',
        'test_json': DATASET_DIR / 'ViTextVQA_test_gt.json',
    },
    'ViOCRVQA': {
        'image_dir': DATASET_DIR / 'data',
        'test_json': DATASET_DIR / 'test.json',
    },
}
dataset_defaults = DEFAULT_DATASET_LAYOUT.get(
    DATASET_NAME,
    {'image_dir': DATASET_DIR / 'images', 'test_json': DATASET_DIR / 'test.json'},
)
IMAGE_DIR = dataset_defaults['image_dir']  # override nếu folder ảnh khác tên
TEST_JSON = dataset_defaults['test_json']  # override nếu file test khác tên

MODEL_VARIANT = '2B'  # '0.8B' | '2B' | '4B' | nhập trực tiếp HF model id
BASE_MODEL_CONFIG = {
    'min_pixels': 256 * 28 * 28,
    'max_pixels': 768 * 28 * 28,
    'default_batch_size': 2,
    'enable_thinking': False,
    'trust_remote_code': False,
    'attn_implementation': None,
    'generation': {
        'do_sample': True,
        'temperature': 0.7,
        'top_p': 0.8,
        'top_k': 20,
        'min_p': 0.0,
        'repetition_penalty': 1.0,
        'max_new_tokens': 32,
        'num_beams': 1,
    },
}
MODEL_REGISTRY = {
    '0.8B': {
        'model_name': 'Qwen/Qwen3.5-0.8B',
        'default_batch_size': 4,
        'max_pixels': 640 * 28 * 28,
    },
    '2B': {
        'model_name': 'Qwen/Qwen3.5-2B',
        'default_batch_size': 2,
        'max_pixels': 768 * 28 * 28,
    },
    '4B': {
        'model_name': 'Qwen/Qwen3.5-4B',
        'default_batch_size': 1,
        'max_pixels': 640 * 28 * 28,
    },
}
selected_model_overrides = MODEL_REGISTRY.get(MODEL_VARIANT, {'model_name': MODEL_VARIANT})
MODEL_CONFIG = {**BASE_MODEL_CONFIG, **selected_model_overrides}
MODEL_CONFIG['generation'] = {
    **BASE_MODEL_CONFIG['generation'],
    **selected_model_overrides.get('generation', {}),
}
MODEL_NAME = MODEL_CONFIG['model_name']

DEVICE = 'auto'  # 'auto' | 'cuda' | 'cpu'
BATCH_SIZE = MODEL_CONFIG['default_batch_size']
MAX_SAMPLES = None        # None = chạy toàn bộ test set
SKIP_MISSING_IMAGES = False
SEED = 42
IMAGE_EXTS = ['.jpg', '.jpeg', '.png', '.webp', '.bmp']
TRANSFORMERS_MIN_VERSION = '5.2.0'

SYSTEM_PROMPT = (
    'Bạn là trợ lý VQA tiếng Việt. '
    'Trả lời dựa trên nội dung nhìn thấy trong ảnh và không bịa thêm thông tin.'
)
USER_PROMPT_TEMPLATE = (
    'Hãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.\n'
    'Câu hỏi: {question}'
)

MODEL_TAG = MODEL_VARIANT.lower().replace('/', '_').replace('.', '_')
OUTPUT_DIR = REPO_ROOT / 'baseline' / 'outputs' / f'qwen35_{DATASET_NAME.lower()}_{MODEL_TAG}'
PREDICTIONS_PATH = OUTPUT_DIR / 'predictions.json'
STATS_PATH = OUTPUT_DIR / 'stats.json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENERATION_CONFIG = MODEL_CONFIG['generation']
CONFIG = {
    'repo_root': str(REPO_ROOT),
    'dataset_name': DATASET_NAME,
    'dataset_dir': str(DATASET_DIR),
    'image_dir': str(IMAGE_DIR),
    'test_json': str(TEST_JSON),
    'image_exts': IMAGE_EXTS,
    'model': {
        'variant': MODEL_VARIANT,
        'model_name': MODEL_NAME,
        'supported_variants': sorted(MODEL_REGISTRY.keys()),
        'min_pixels': MODEL_CONFIG['min_pixels'],
        'max_pixels': MODEL_CONFIG['max_pixels'],
        'default_batch_size': MODEL_CONFIG['default_batch_size'],
        'enable_thinking': MODEL_CONFIG['enable_thinking'],
        'trust_remote_code': MODEL_CONFIG['trust_remote_code'],
        'attn_implementation': MODEL_CONFIG['attn_implementation'],
    },
    'generation': GENERATION_CONFIG,
    'runtime': {
        'device': DEVICE,
        'batch_size': BATCH_SIZE,
        'max_samples': MAX_SAMPLES,
        'skip_missing_images': SKIP_MISSING_IMAGES,
        'seed': SEED,
        'transformers_min_version': TRANSFORMERS_MIN_VERSION,
    },
    'outputs': {
        'output_dir': str(OUTPUT_DIR),
        'predictions_path': str(PREDICTIONS_PATH),
        'stats_path': str(STATS_PATH),
    },
}
CONFIG


In [ ]:
import json
import random
import re
import unicodedata
from collections import Counter
from functools import lru_cache
from importlib import import_module
from importlib.metadata import PackageNotFoundError, version as package_version

import numpy as np
import torch
from packaging import version
from PIL import Image
from tqdm.auto import tqdm

try:
    current_transformers_version = package_version('transformers')
except PackageNotFoundError as exc:
    raise RuntimeError(
        'Notebook này cần transformers>=5.2.0. Hãy cài bằng: '
        "%pip install -q -U 'transformers>=5.2.0' accelerate huggingface_hub pillow tqdm"
    ) from exc

if version.parse(current_transformers_version) < version.parse(TRANSFORMERS_MIN_VERSION):
    raise RuntimeError(
        f'transformers={current_transformers_version} quá cũ. '
        f'Cần >={TRANSFORMERS_MIN_VERSION}. '
        "Chạy: %pip install -q -U 'transformers>=5.2.0' accelerate huggingface_hub pillow tqdm"
    )

transformers_module = import_module('transformers')
AutoProcessor = getattr(transformers_module, 'AutoProcessor')

def resolve_qwen35_model_class():
    for class_name in (
        'Qwen3_5ForConditionalGeneration',
        'AutoModelForImageTextToText',
        'AutoModelForVision2Seq',
    ):
        model_class = getattr(transformers_module, class_name, None)
        if model_class is not None:
            return class_name, model_class
    raise ImportError(
        'Không tìm thấy class load model cho Qwen3.5 trong transformers hiện tại.'
    )

MODEL_CLASS_NAME, Qwen35ModelClass = resolve_qwen35_model_class()

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print({
    'seed': SEED,
    'transformers_version': current_transformers_version,
    'model_class': MODEL_CLASS_NAME,
})


In [ ]:
ZERO_WIDTH_RE = re.compile(r'[\u200b\u200c\u200d\ufeff]')
EDGE_PUNCT_RE = re.compile(r"^[\s\.,!?;:'\"“”‘’`~_]+|[\s\.,!?;:'\"“”‘’`~_]+$")
DASH_TRANSLATION = str.maketrans({'–': '-', '—': '-', '−': '-', '‐': '-'})
SIMPLE_NUMBER_WORDS = {
    'không': '0',
    'một': '1',
    'hai': '2',
    'ba': '3',
    'bốn': '4',
    'tư': '4',
    'năm': '5',
    'lăm': '5',
    'sáu': '6',
    'bảy': '7',
    'bẩy': '7',
    'tám': '8',
    'chín': '9',
    'mười': '10',
    'zero': '0',
    'one': '1',
    'two': '2',
    'three': '3',
    'four': '4',
    'five': '5',
    'six': '6',
    'seven': '7',
    'eight': '8',
    'nine': '9',
    'ten': '10',
}

def _canonicalize_number(text: str) -> str:
    compact = text.replace(' ', '')
    if re.fullmatch(r'\d{1,3}([.,]\d{3})+', compact):
        return re.sub(r'[.,]', '', compact)
    if re.fullmatch(r'\d+,\d+', compact) and not re.fullmatch(r'\d{1,3}(,\d{3})+', compact):
        return compact.replace(',', '.')
    return text

def normalize_answer(text: str) -> str:
    if text is None:
        return ''
    text = str(text)
    text = unicodedata.normalize('NFC', text)
    text = text.translate(DASH_TRANSLATION)
    text = ZERO_WIDTH_RE.sub('', text)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = EDGE_PUNCT_RE.sub('', text)
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF\-\./,]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if text in SIMPLE_NUMBER_WORDS:
        text = SIMPLE_NUMBER_WORDS[text]
    text = _canonicalize_number(text)
    text = re.sub(r'^[\.,!?;:]+|[\.,!?;:]+$', '', text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def exact_match_score(prediction: str, ground_truths: list[str]) -> float:
    pred = normalize_answer(prediction)
    return float(any(normalize_answer(gt) == pred for gt in ground_truths))

def _token_f1_single(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(gt_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def f1_score(prediction: str, ground_truths: list[str]) -> float:
    return max((_token_f1_single(prediction, gt) for gt in ground_truths), default=0.0)

def compute_metrics(predictions: list[str], ground_truths: list[list[str]]) -> dict:
    assert len(predictions) == len(ground_truths)
    if not predictions:
        return {'exact_match': 0.0, 'f1': 0.0, 'num_samples': 0}
    em = sum(exact_match_score(pred, gts) for pred, gts in zip(predictions, ground_truths))
    f1 = sum(f1_score(pred, gts) for pred, gts in zip(predictions, ground_truths))
    n = len(predictions)
    return {
        'exact_match': em / n,
        'f1': f1 / n,
        'num_samples': n,
    }

def load_annotations(json_path: Path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ['annotations', 'data', 'items', 'samples']:
            if key in data and isinstance(data[key], list):
                return data[key]
    raise ValueError(f'Unsupported annotation format: {json_path}')

def extract_question(ann: dict) -> str:
    for key in ['question', 'query', 'text']:
        if ann.get(key):
            return str(ann[key])
    raise KeyError(f'Cannot find question field in sample: {ann}')

def extract_ground_truths(ann: dict) -> list[str]:
    if isinstance(ann.get('answers'), list):
        return [str(x) for x in ann['answers']]
    if isinstance(ann.get('all_answers'), list):
        return [str(x) for x in ann['all_answers']]
    if ann.get('answer') is not None:
        return [str(ann['answer'])]
    if ann.get('label') is not None:
        if isinstance(ann['label'], list):
            return [str(x) for x in ann['label']]
        return [str(ann['label'])]
    return ['']

def extract_image_identifier(ann: dict) -> str:
    for key in ['image_id', 'image', 'image_name', 'file_name', 'filename']:
        value = ann.get(key)
        if value is not None:
            return str(value)
    raise KeyError(f'Cannot find image identifier field in sample: {ann}')

@lru_cache(maxsize=100000)
def resolve_image_path(image_identifier: str, image_dir: str, image_exts: tuple[str, ...]):
    image_dir = Path(image_dir)
    candidate = Path(image_identifier)
    if candidate.suffix:
        direct = image_dir / candidate.name
        if direct.exists():
            return direct
        if candidate.exists():
            return candidate
    stem = candidate.stem if candidate.suffix else candidate.name
    for ext in image_exts:
        path = image_dir / f'{stem}{ext}'
        if path.exists():
            return path
    matches = sorted(image_dir.glob(f'{stem}.*'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Cannot find image for id '{image_identifier}' in {image_dir}")

def build_prompt_for_log(question: str) -> str:
    return '\n'.join([
        '[SYSTEM]',
        SYSTEM_PROMPT.strip(),
        '',
        '[USER]',
        USER_PROMPT_TEMPLATE.format(question=question).strip(),
    ])

print(normalize_answer('  Một—nghìn  '))
print(normalize_answer('1.000'))


In [ ]:
if DEVICE == 'auto':
    resolved_device = 'cuda' if torch.cuda.is_available() else 'cpu'
else:
    resolved_device = DEVICE

if resolved_device == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda' nhưng máy hiện tại không có CUDA.")

use_device_map = DEVICE == 'auto' and resolved_device == 'cuda'
torch_dtype = torch.float32 if resolved_device == 'cpu' else (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

print(f'Loading processor: {MODEL_NAME}')
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MODEL_CONFIG['min_pixels'],
    max_pixels=MODEL_CONFIG['max_pixels'],
    trust_remote_code=MODEL_CONFIG['trust_remote_code'],
)
if hasattr(processor, 'tokenizer') and processor.tokenizer is not None:
    if processor.tokenizer.pad_token_id is None and processor.tokenizer.eos_token is not None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = 'left'

load_kwargs = {
    'torch_dtype': torch_dtype,
    'trust_remote_code': MODEL_CONFIG['trust_remote_code'],
}
if use_device_map:
    load_kwargs['device_map'] = 'auto'
if MODEL_CONFIG.get('attn_implementation'):
    load_kwargs['attn_implementation'] = MODEL_CONFIG['attn_implementation']

print(f'Loading model: {MODEL_NAME}')
model = Qwen35ModelClass.from_pretrained(MODEL_NAME, **load_kwargs)
if not use_device_map:
    model = model.to(resolved_device)
model.eval()

target_device = getattr(model, 'device', None)
if target_device is None:
    target_device = next(model.parameters()).device

pad_token_id = None
if hasattr(processor, 'tokenizer') and processor.tokenizer is not None:
    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = processor.tokenizer.eos_token_id

print({
    'resolved_device': str(target_device),
    'torch_dtype': str(torch_dtype),
    'use_device_map': use_device_map,
    'pad_token_id': pad_token_id,
    'enable_thinking': MODEL_CONFIG['enable_thinking'],
    'generation': GENERATION_CONFIG,
})


## Inference Helpers

Batching bên dưới chuyển sang dùng trực tiếp `processor.apply_chat_template(...)` của `Qwen3.5`.
Notebook cố gắng truyền `enable_thinking=False` khi render prompt; nếu bản `transformers` hiện tại dùng API khác cho chat-template kwargs thì notebook sẽ fallback tự động.

In [ ]:
def build_messages(question: str, image: Image.Image) -> list[dict]:
    messages = []
    if SYSTEM_PROMPT.strip():
        messages.append({
            'role': 'system',
            'content': [{'type': 'text', 'text': SYSTEM_PROMPT.strip()}],
        })
    messages.append({
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': USER_PROMPT_TEMPLATE.format(question=question).strip()},
        ],
    })
    return messages

def apply_qwen35_chat_template(conversations: list[list[dict]]):
    common_kwargs = {
        'tokenize': True,
        'add_generation_prompt': True,
        'return_dict': True,
        'return_tensors': 'pt',
        'padding': True,
    }
    attempts = [
        {'enable_thinking': MODEL_CONFIG['enable_thinking']},
        {'chat_template_kwargs': {'enable_thinking': MODEL_CONFIG['enable_thinking']}},
        {},
    ]
    last_error = None
    for extra_kwargs in attempts:
        try:
            return processor.apply_chat_template(conversations, **common_kwargs, **extra_kwargs)
        except TypeError as exc:
            last_error = exc
    raise last_error

def move_to_device(inputs: dict, device: torch.device) -> dict:
    return {
        k: (v.to(device) if isinstance(v, torch.Tensor) else v)
        for k, v in inputs.items()
    }

def build_generate_kwargs() -> dict:
    kwargs = {
        'max_new_tokens': GENERATION_CONFIG['max_new_tokens'],
        'num_beams': GENERATION_CONFIG['num_beams'],
        'repetition_penalty': GENERATION_CONFIG['repetition_penalty'],
        'pad_token_id': pad_token_id,
    }
    if GENERATION_CONFIG['do_sample']:
        kwargs.update({
            'do_sample': True,
            'temperature': GENERATION_CONFIG['temperature'],
            'top_p': GENERATION_CONFIG['top_p'],
            'top_k': GENERATION_CONFIG['top_k'],
            'min_p': GENERATION_CONFIG['min_p'],
        })
    else:
        kwargs['do_sample'] = False
    return kwargs

@torch.no_grad()
def generate_batch(batch_items: list[dict]) -> list[str]:
    conversations = [build_messages(item['question'], item['image']) for item in batch_items]
    inputs = apply_qwen35_chat_template(conversations)
    inputs = move_to_device(dict(inputs), target_device)
    prompt_length = inputs['input_ids'].shape[1]
    generate_kwargs = build_generate_kwargs()
    try:
        generated = model.generate(**inputs, **generate_kwargs)
    except TypeError as exc:
        if 'min_p' not in str(exc):
            raise
        print('Warning: transformers hiện tại chưa hỗ trợ min_p trong generate(); bỏ min_p và chạy lại.')
        generate_kwargs.pop('min_p', None)
        generated = model.generate(**inputs, **generate_kwargs)
    return [
        processor.decode(seq[prompt_length:], skip_special_tokens=True).strip()
        for seq in generated
    ]


In [ ]:
annotations = load_annotations(TEST_JSON)


In [ ]:
annotations = load_annotations(TEST_JSON)
if MAX_SAMPLES is not None:
    annotations = annotations[:MAX_SAMPLES]

print(f'Loaded {len(annotations):,} test samples from {TEST_JSON}')

results = []
skipped = []
image_exts_tuple = tuple(IMAGE_EXTS)

for start in tqdm(range(0, len(annotations), BATCH_SIZE), desc='Running baseline'):
    batch_anns = annotations[start:start + BATCH_SIZE]
    batch_items = []

    for ann in batch_anns:
        question = extract_question(ann)
        ground_truths = extract_ground_truths(ann)
        image_identifier = extract_image_identifier(ann)
        question_id = ann.get('id', ann.get('question_id', len(results)))

        try:
            image_path = resolve_image_path(image_identifier, str(IMAGE_DIR), image_exts_tuple)
        except FileNotFoundError as exc:
            if SKIP_MISSING_IMAGES:
                skipped.append({
                    'question_id': question_id,
                    'image_identifier': image_identifier,
                    'error': str(exc),
                })
                continue
            raise

        image = Image.open(image_path).convert('RGB')
        batch_items.append({
            'question_id': question_id,
            'image_id': ann.get('image_id', image_identifier),
            'image_path': str(image_path),
            'question': question,
            'ground_truths': ground_truths,
            'image': image,
            'prompt': build_prompt_for_log(question),
        })

    if not batch_items:
        continue

    try:
        predictions = generate_batch(batch_items)
    finally:
        for item in batch_items:
            item['image'].close()

    for item, prediction in zip(batch_items, predictions):
        em = exact_match_score(prediction, item['ground_truths'])
        f1 = f1_score(prediction, item['ground_truths'])
        results.append({
            'question_id': item['question_id'],
            'image_id': item['image_id'],
            'image_path': item['image_path'],
            'question': item['question'],
            'prompt': item['prompt'],
            'prediction': prediction,
            'prediction_normalized': normalize_answer(prediction),
            'ground_truths': item['ground_truths'],
            'ground_truths_normalized': [normalize_answer(x) for x in item['ground_truths']],
            'exact_match': em,
            'f1': f1,
        })

predictions = [x['prediction'] for x in results]
ground_truths = [x['ground_truths'] for x in results]
metrics = compute_metrics(predictions, ground_truths)

stats = {
    'dataset_name': DATASET_NAME,
    'dataset_dir': str(DATASET_DIR),
    'test_json': str(TEST_JSON),
    'image_dir': str(IMAGE_DIR),
    'model_variant': MODEL_VARIANT,
    'model_name': MODEL_NAME,
    'model_class': MODEL_CLASS_NAME,
    'device': DEVICE,
    'resolved_device': str(target_device),
    'transformers_version': current_transformers_version,
    'generation': GENERATION_CONFIG,
    'enable_thinking': MODEL_CONFIG['enable_thinking'],
    'metrics': metrics,
    'num_predictions': len(results),
    'num_skipped': len(skipped),
    'predictions_path': str(PREDICTIONS_PATH),
    'stats_path': str(STATS_PATH),
}

with open(PREDICTIONS_PATH, 'w', encoding='utf-8') as f:
    json.dump({'stats': stats, 'predictions': results, 'skipped': skipped}, f, ensure_ascii=False, indent=2)

with open(STATS_PATH, 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(json.dumps(stats, ensure_ascii=False, indent=2))
print(f'Saved predictions -> {PREDICTIONS_PATH}')
print(f'Saved stats -> {STATS_PATH}')
results[:3]


In [ ]:
stats
